# Next Lab — Kaggle Free GPU LoRA Fine-tuning

This notebook runs a bounded LoRA fine-tuning experiment on a **Kaggle GPU session**. It uses the small `distilgpt2` base model. Select a free GPU accelerator in Notebook Settings before running.

**Cost guard:** use only Kaggle's free quota. Do not enable paid products or external paid compute. Free GPU availability, accelerator type, and quota are controlled by Kaggle and are not guaranteed.

## 1. Enable a Kaggle GPU

In the notebook's right-side **Settings** panel, set Accelerator to **GPU** (choose an available free option such as T4 or P100 if offered). Turn Internet on so the notebook can download model weights and packages. Save the settings, then run the next cell. This notebook stops if CUDA is unavailable.

In [ ]:
!pip -q install -U "transformers>=4.45,<5" "datasets>=3,<4" "peft>=0.14,<1" accelerate

import os, torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Kaggle Notebook Settings choose Accelerator = GPU, save, and restart the session.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2. Load a dataset from Next Lab or Kaggle

Recommended: open Next Lab → Training, choose a ready dataset, and copy its temporary signed download URL (valid for about one hour). Paste it below. Or leave the URL blank and attach a CSV/JSONL file as a Kaggle Dataset; the notebook will look under `/kaggle/input`.

The file must be CSV with a `text` column or JSONL with a `text` field. Use only data you are authorized to process.

In [ ]:
import os, glob, getpass, requests
from pathlib import Path
signed_url = getpass.getpass("Optional: paste the Next Lab signed dataset URL (leave blank to use a Kaggle Input Dataset): ").strip()
data_path = None
if signed_url:
    response = requests.get(signed_url, timeout=120)
    response.raise_for_status()
    suffix = ".jsonl" if ".jsonl" in signed_url.lower() else ".csv"
    data_path = "/kaggle/working/next_lab_dataset" + suffix
    Path(data_path).write_bytes(response.content)
else:
    candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True) + glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
    if not candidates:
        raise FileNotFoundError("No CSV/JSONL file found. Attach a Kaggle Dataset or rerun and paste a Next Lab signed URL.")
    data_path = candidates[0]
print("Dataset:", data_path)

In [ ]:
from datasets import load_dataset
if data_path.lower().endswith(".jsonl"):
    ds = load_dataset("json", data_files=data_path, split="train")
elif data_path.lower().endswith(".csv"):
    ds = load_dataset("csv", data_files=data_path, split="train")
else:
    raise ValueError("Upload CSV or JSONL.")
if "text" not in ds.column_names:
    raise ValueError(f"Dataset requires a text column/field. Found: {ds.column_names}")
ds = ds.filter(lambda row: isinstance(row["text"], str) and bool(row["text"].strip()))
if not len(ds): raise ValueError("No non-empty text rows found.")
MAX_TRAIN_ROWS = 1000
ds = ds.select(range(min(len(ds), MAX_TRAIN_ROWS)))
print("Training rows:", len(ds), "| sample:", ds[0]["text"][:200])

## 3. Configure the model and LoRA

This starter is intentionally fixed to `distilgpt2` (~82M parameters) for a modest first GPU run. It is fine-tuning, not pretraining a new foundation model. Keep the base model ID with the adapter artifact.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL = "distilgpt2"
OUTPUT_DIR = "/kaggle/working/next-lab-lora"
MAX_LENGTH = 256
MAX_STEPS = 100
BATCH_SIZE = 2
GRAD_ACCUM = 4

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
model.config.use_cache = False
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["c_attn", "c_proj"], fan_in_fan_out=True)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Device:", next(model.parameters()).device)

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
args = TrainingArguments(
    output_dir=OUTPUT_DIR, max_steps=MAX_STEPS, per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=1e-4, warmup_steps=5,
    logging_steps=10, save_strategy="no", fp16=True, dataloader_num_workers=2,
    report_to="none", optim="adamw_torch", gradient_checkpointing=True
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))

## 4. Start training

Run this cell only when you are ready. Kaggle's free session can stop or disconnect; save/download the artifact promptly after training.

In [ ]:
result = trainer.train()
print(result.metrics)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved to", OUTPUT_DIR)

## 5. Export the LoRA adapter

Download this ZIP from the notebook output. It contains adapter weights/configuration and tokenizer files, not the full base model. Next Lab's automatic artifact-import and job-completion callback is not yet connected, so keep the job marked as manually run until the artifact is imported and verified.

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/next-lab-lora-adapter", "zip", OUTPUT_DIR)
print("Download from Kaggle Output files:", archive)

## Important limits

- Kaggle free GPU quota and accelerator availability vary by account, region, and time.
- This is a small LoRA fine-tuning starter, not full model pretraining or production-scale training.
- Keep the `distilgpt2` base model identifier with the adapter.
- This notebook does not automatically mark a Next Lab job complete or upload the adapter into Supabase. That requires a separate authenticated artifact-import feature.
- Do not paste Supabase service-role keys or other secrets into a notebook.